In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "utils" / "paths.py").is_file()), None)
if PROJECT_ROOT is None:
    raise RuntimeError("Start the notebook inside the project directory")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from utils.paths import RAW, stage_path
from utils.runs import require_stage, stage_run
from loguru import logger
from rdkit import Chem
from rdkit.Chem import Lipinski, Descriptors
from pathlib import Path
from loguru import logger
from tqdm.auto import tqdm
import io
import os

In [ ]:
input_filename = require_stage("01_generation") / "generated_ligands.smi"
BATCH_SIZE = 10000

In [ ]:
def check_lipinski(mol):
    if mol is None:
        return False, "INVALID_SMILES"
        
    mw = Descriptors.MolWt(mol)
    logp = Descriptors.MolLogP(mol)
    hbd = Lipinski.NumHDonors(mol)
    hba = Lipinski.NumHAcceptors(mol)

    if mw > 500:
        return False, "MW > 500"
    if logp > 5:
        return False, "LogP > 5"
    if hbd > 5:
        return False, "HBD > 5"
    if hba > 10:
        return False, "HBA > 10"

    return True, None

In [ ]:
with stage_run("02_lipinski", [input_filename], {}) as output_dir:
    output_filename = output_dir / "lipinski_passed.smi"
    stats = {
        "Total": 0,
        "Passed": 0,
        "MW > 500": 0,
        "LogP > 5": 0,
        "HBD > 5": 0,
        "HBA > 10": 0
    }

    logger.info(f"Starting filtering (Lipinski): {input_filename} -> {output_filename}")
    BATCH_SIZE = 10000 

    buffer = io.StringIO()
    counter_written = 0

    with open(input_filename, "r") as f_in, open(output_filename, "w") as f_out:
        stats["Total"] = 0
    
        for line in tqdm(f_in, desc="Processing Lipinski filter"):
            stats["Total"] += 1
            parts = line.strip().split(maxsplit=1)
            if len(parts) < 2:
                stats["Malformed"] = stats.get("Malformed", 0) + 1
                continue
            smiles, comb = parts[0], parts[1]
        
            mol = Chem.MolFromSmiles(smiles)
        
            is_pass, reason = check_lipinski(mol)

            if is_pass:
                counter_written += 1
                buffer.write(f"""{smiles} {comb}\n""")
                stats["Passed"] += 1
            
                if counter_written % BATCH_SIZE == 0:
                    f_out.write(buffer.getvalue())
                    buffer.seek(0)
                    buffer.truncate(0)
            else:
                stats[reason] = stats.get(reason, 0) + 1

        f_out.write(buffer.getvalue())

    logger.success("Filtering completed!")
    logger.info(f"Results saved to: {output_filename}")
    logger.info("-" * 30)
    logger.info(f"Total processed: {stats['Total']}")
    if stats['Total'] > 0:
        logger.info(f"Passed:  {stats['Passed']} ({(stats['Passed']/stats['Total'])*100:.1f}%)")
    logger.info("-" * 30)
    logger.info("Rejection reasons:")
    for reason, count in stats.items():
        if reason not in ["Total", "Passed"] and count > 0:
            logger.info(f"{reason}: {count}")
